#task1

In [ ]:
from langchain.document_loaders import TextLoader

# Load document
loader = TextLoader("sample.txt")
documents = loader.load()

# Extract text
text = documents[0].page_content

# Total characters
print("Total Characters:", len(text))

# Preview content
print("\n--- Sample Preview ---\n")
print(text[:500])

#task 2

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(temperature=0)

# Prompt template
template = """
You are a helpful assistant that summarizes text.

Text:
{text}

Summary:
"""

prompt = PromptTemplate(
    input_variables=["text"],
    template=template
)

# Generate summary
formatted_prompt = prompt.format(text=text)
summary = llm.predict(formatted_prompt)

print("\n--- Summary ---\n")
print(summary)

#task 3

In [ ]:
short_template = """
Summarize the following text in 5-6 lines:

{text}
"""

short_prompt = PromptTemplate(
    input_variables=["text"],
    template=short_template
)

short_summary = llm.predict(short_prompt.format(text=text))

print("\n--- Short Summary ---\n")
print(short_summary)

#task 4

In [ ]:
bullet_template = """
Summarize the following text into bullet points:

{text}
"""

bullet_prompt = PromptTemplate(
    input_variables=["text"],
    template=bullet_template
)

bullet_summary = llm.predict(bullet_prompt.format(text=text))

print("\n--- Bullet Summary ---\n")
print(bullet_summary)

#task 5

In [ ]:
from langchain.chains.summarize import load_summarize_chain
from langchain.chat_models import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(temperature=0)

# Load Stuff Chain
stuff_chain = load_summarize_chain(llm, chain_type="stuff")

# Generate summary
summary = stuff_chain.run(documents)

print("\n--- Stuff Chain Summary ---\n")
print(summary)

# 📄 Task 6 & 7 – LangChain Summarization

---

## ✅ Task 6: Comparison with Prompt-Based Summary

### 🔹 Objective

Compare:

* Prompt-based summarization
* Stuff chain summarization

---

### 🔹 Comparison Table

| Feature              | Prompt-Based Summary       | Stuff Chain Summary      |
| -------------------- | -------------------------- | ------------------------ |
| Approach             | Manual prompt              | Built-in chain (`stuff`) |
| Flexibility          | High (custom instructions) | Medium                   |
| Ease of Use          | Moderate                   | Easy                     |
| Custom Formatting    | Fully customizable         | Limited                  |
| Token Limit Handling | ❌ Not handled              | ❌ Not handled            |
| Best For             | Controlled outputs         | Quick summarization      |

---

### 🔹 Differences in Output Quality

* **Prompt-Based Summary**

  * Can be tailored (e.g., tone, length, format)
  * Produces more structured and specific output
  * Depends heavily on prompt design

* **Stuff Chain Summary**

  * Produces more generic summaries
  * Less control over formatting
  * Faster to implement

---

### 🔹 Limitations

* Both approaches:

  * ❌ Not suitable for large documents
  * ❌ Limited by token size
  * ❌ May lose important details in long text

---

## ✅ Task 7: Why Map-Reduce is Needed (Conceptual)

### 🔹 Why Large Documents Need Map-Reduce

* Large documents often exceed LLM token limits
* Cannot be processed in a single request
* Leads to incomplete or failed summarization

👉 Map-Reduce solves this by splitting the document into manageable chunks

---

### 🔹 How Map-Reduce Works

#### 🟢 Map Step

* Divide document into smaller chunks
* Each chunk is summarized independently

#### 🔵 Reduce Step

* Combine all chunk summaries
* Generate a final, coherent summary

---

### 🔹 Key Advantages

* Handles large-scale documents efficiently
* Avoids token limit issues
* Produces more scalable and organized summaries

---


task 8

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.summarize import load_summarize_chain
from langchain.chat_models import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(temperature=0)

# Split text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

# Load Map-Reduce Chain
map_reduce_chain = load_summarize_chain(llm, chain_type="map_reduce")

# Generate summary
final_summary = map_reduce_chain.run(docs)

print("\n--- Map-Reduce Summary ---\n")
print(final_summary)

#task 9

In [ ]:
# Map step outputs (intermediate summaries)
map_chain = load_summarize_chain(llm, chain_type="map_reduce", return_intermediate_steps=True)

result = map_chain({"input_documents": docs}, return_only_outputs=True)

print("\n--- Intermediate Summaries ---\n")
for i, summary in enumerate(result["intermediate_steps"]):
    print(f"\nChunk {i+1} Summary:\n", summary)

#task 10

# ✅ Task 10: Refine Chain (Conceptual)

### 🔹 How Refine Summarization Works

* Processes the document **chunk by chunk**
* Starts with an **initial summary** from the first chunk
* Iteratively **refines the summary** by adding new chunks
* Each step improves and updates the previous summary

👉 Think of it as *progressively improving a draft summary*

---

### 🔹 Difference: Map-Reduce vs Refine

| Feature      | Map-Reduce                    | Refine                    |
| ------------ | ----------------------------- | ------------------------- |
| Processing   | Parallel (independent chunks) | Sequential (step-by-step) |
| Intermediate | Multiple summaries            | Single evolving summary   |
| Speed        | Faster                        | Slower                    |
| Coherence    | Moderate                      | High                      |
| Context Flow | Weak between chunks           | Strong across chunks      |

---

### 🔹 Key Insight

* **Map-Reduce** → Best for speed & large-scale processing
* **Refine** → Best for **high-quality, coherent summaries**


#task 11

In [ ]:
from langchain.chains.summarize import load_summarize_chain
from langchain.chat_models import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(temperature=0)

# Load Refine Chain
refine_chain = load_summarize_chain(llm, chain_type="refine")

# Generate refined summary
refined_summary = refine_chain.run(docs)

print("\n--- Refine Summary ---\n")
print(refined_summary)

#task 12

# ✅ Task 12: Comparison of Summarization Methods

### 🔹 Comparison Table

| Method       | Quality     | Coherence | Long Docs Suitability | Speed  |
| ------------ | ----------- | --------- | --------------------- | ------ |
| Prompt-Based | Medium-High | Medium    | ❌ No                  | Fast   |
| Stuff Chain  | Medium      | Medium    | ❌ No                  | Fast   |
| Map-Reduce   | High        | Medium    | ✅ Yes                 | Medium |
| Refine Chain | Very High   | High      | ✅ Yes                 | Slow   |

---

### 🔹 Evaluation

* **Best Quality** → Refine Chain
* **Best for Large Docs** → Map-Reduce / Refine
* **Fastest** → Prompt-based / Stuff
* **Best Balance** → Map-Reduce

---

### 🔹 Conclusion

* Use **Refine** when accuracy matters
* Use **Map-Reduce** for scalability
* Use **Stuff/Prompt** for small inputs


#task 13

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain
from langchain.chat_models import ChatOpenAI

def summarize_document(text, method="map_reduce"):
    llm = ChatOpenAI(temperature=0)

    if method == "prompt":
        template = "Summarize the following text:\n{text}"
        prompt = PromptTemplate(input_variables=["text"], template=template)
        return llm.predict(prompt.format(text=text))

    elif method == "stuff":
        chain = load_summarize_chain(llm, chain_type="stuff")
        return chain.run(documents)

    elif method == "map_reduce":
        chain = load_summarize_chain(llm, chain_type="map_reduce")
        return chain.run(docs)

    elif method == "refine":
        chain = load_summarize_chain(llm, chain_type="refine")
        return chain.run(docs)

    else:
        return "Invalid method"

#task 14

# ✅ Task 14: Observations & Insights

### 🔹 Best Strategy for Very Long Documents

* **Refine Chain** → Best for accuracy & coherence
* **Map-Reduce** → Best for handling very large data efficiently

---

### 🔹 Trade-offs: Speed vs Quality

| Method     | Speed  | Quality   |
| ---------- | ------ | --------- |
| Prompt     | Fast   | Medium    |
| Stuff      | Fast   | Medium    |
| Map-Reduce | Medium | High      |
| Refine     | Slow   | Very High |

---

### 🔹 Final Insight

* Faster methods → Less detailed summaries
* Slower methods → Better understanding and coherence

👉 Choose based on your use case:

* Speed → Prompt / Stuff
* Balance → Map-Reduce
* Accuracy → Refine
